# Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xo_env

from dqn import DQNAgent
from show_video import show_video
from video_maker import VideoMaker
import object_cache

from rule_based_agent import RuleBasedAgent

# Env Exploration

In [ ]:
env = xo_env.XO_Env()

In [ ]:
s = env.reset(randomGrid=False)
s

In [ ]:
env.getStateShape()

In [ ]:
env.getActionsCount()

# Random Walk

In [ ]:
def randomWalk():
    frames = []
    s = env.reset(randomGrid=False)
    d = False
    
    while d == False:
        frames.append(env.render())
        a = np.random.randint(env.getActionsCount())
        s,d,r = env.step(a)
        
    frames.append(env.render())  
    return frames

In [ ]:
for i in range(1):
    video = VideoMaker(f"random_walk_{i:2d}", fps=6)
    video.extend(randomWalk())
    video.export()
    show_video(video.path())

# Build Agent

In [ ]:
n_action = env.getActionsCount()
state_shape = env.getStateShape()

In [ ]:
cache_epsilon = object_cache.load("cache_epsilon", lambda: 0.99)

In [ ]:
cache_epsilon 

In [ ]:
agent = DQNAgent(
    state_shape, 
    n_action, 
    gamma = 0.95, 
    epsilon= cache_epsilon,
    cache_weights_name="xo_agent",
    replay_buffer_size = 150)

In [ ]:
agent.summary()

In [ ]:
agent.plot_model(dpi = 90)

# Train Agent

In [ ]:
return_history = object_cache.load("return_history_list", lambda: [])

In [ ]:
n_episodes = 3500
n_max_allowed_steps = 11
update_rate = 5
batch_size = 15

In [ ]:
def try_greedy_policy(episode_no, export=True):
    s = env.reset(randomGrid=False)
    d = False
    total_return = 0
    
    steps_counter = 0
    
    video = VideoMaker(f"Agent after {episode_no} episodes", resize_shape= (500,500), fps=6)
        
    while not d and (steps_counter < n_max_allowed_steps):
        video.append(env.render())
        video.setText(f"Return: {total_return:+05d}, Episode: {episode_no:04d}", color=(0,0,0),txt_position=(3,30))
        
        a = agent.act(np.array([s]))
        s, d, r= env.step(a)
        
        total_return += r
        
        steps_counter+=1

    for i in range(5):
        video.append(env.render())
        video.setText(f"Return: {total_return:+05d}, Episode: {episode_no:04d}", color=(0,0,0),txt_position=(3,30))
    
    if export==True:
        show_video(video.export())
        
    return total_return, video

In [ ]:
def train(env, agent, n_episodes, n_max_allowed_steps, update_rate, batch_size):
    global return_history
    steps_counter = 0
    
    for episode in range(len(return_history), n_episodes):     
        s = env.reset(randomGrid= True)
        total_return = 0
        n_target_update = 0
        
        for i in range(n_max_allowed_steps):
            print(f"Episode: {episode + 1}, Steps: {i+1}/{n_max_allowed_steps}, Total_Return: {total_return :0.2f}, N Updates: {n_target_update}"+ ' '*20,end='\r')
            
            # Force ai agent learn from rule based agent acions
            if np.random.rand() < 0.5:
                a = agent.epsilon_greedy_policy(np.array([s]))
            else:
                a = RuleBasedAgent().act(env.getGrid(), env.getCurrentPlayer())
                if a is None or a < 0:
                    a = agent.epsilon_greedy_policy(np.array([s]))
            
            s_, d,r = env.step(a)
            
            
            total_return += r
            agent.store_transiton(np.array([s]), a, r, np.array([s_]), d)
            
            s = s_
            steps_counter+=1
            
            agent.update_target_network_soft()
            
            if steps_counter % update_rate == 0:
                agent.update_target_network_hard()
                n_target_update += 1
            
            if len(agent.replay_buffer) > batch_size:
                agent.train(batch_size)
                
            if d:
                break
        
        return_history.append(total_return)
        agent.save_nn()
        object_cache.save(return_history,'return_history_list')
        
        if episode % 5 == 0:
            try_greedy_policy(episode + 1)
        agent.update_epsilon()
        object_cache.save(agent.get_epsilion(), "cache_epsilon")
            
        print(f'Episode: {episode + 1}, Steps: {i+1}, Total Return: {total_return :0.2f}, N Updates: {n_target_update}, Epsilion: {agent.get_epsilion():0.3f}' + ' '*20, end='\n')
        
    print('Training Complete')

In [ ]:
train(env, agent, n_episodes, n_max_allowed_steps, update_rate, batch_size)

In [ ]:
plt.scatter(y=return_history, x=list(range(0,len(return_history))))

In [ ]:
plt.plot(return_history)

# Test agent

In [ ]:
def try_agent(title, export=True):
    s = env.reset(randomGrid=False)
    d = False
    total_return = 0
    
    steps_counter = 0
    
    video = VideoMaker(title, resize_shape= (500,500), fps=6)
    
    while not d and steps_counter < n_max_allowed_steps:
        video.append(env.render())
        video.setText(f"Return: {total_return:+05d}, Testing", color=(0,0,0),txt_position=(3,30))
        
        a = agent.act(np.array([s]))
        s, d, r= env.step(a)
        
        total_return += r
        
        steps_counter+=1

    for i in range(5):
        video.append(env.render())
        video.setText(f"Return: {total_return:+05d}, Testing", color=(0,0,0),txt_position=(3,30))
    
    if export==True:
        show_video(video.export())
        
    return total_return, video

In [ ]:
for i in range(1):
    try_agent(f"Agent test no {i}")